<div style="background: linear-gradient(135deg, #4B2E83 0%, #32006E 100%); padding: 40px 30px; border-radius: 15px; box-shadow: 0 8px 20px rgba(75,46,131,0.3);">
  <img src="https://uw-s3-cdn.s3.us-west-2.amazonaws.com/wp-content/uploads/sites/230/2023/11/02134828/Signature_Left_Gold_RGB.png" alt="University of Washington" style="height: 80px; margin-bottom: 20px;">
  <h1 style="color: #FFF; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; font-size: 2.8em; margin: 20px 0 15px 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">Multidimensional Scaling &amp; Autoencoders</h1>
  <p style="color: #E8D3FF; font-family: 'Encode Sans', 'Arial', sans-serif; font-size: 1.3em; margin: 10px 0; font-weight: 300;">BDATA 412 — Advanced Data Visualization · Section 7</p>
  <p style="color: #B7A6D5; font-family: 'Encode Sans', 'Arial', sans-serif; font-size: 1.1em; margin: 5px 0;">University of Washington</p>
</div>

<div style="background-color: #F5F5F5; padding: 15px 25px; border-left: 5px solid #4B2E83; border-radius: 5px; margin-top: 25px;">
  <p style="font-size: 1.1em; color: #333; margin: 0;"><strong style="color: #4B2E83;">Open in Google Colab:</strong></p>
  <p style="margin: 10px 0 5px 0;"><a target="_blank" href="https://colab.research.google.com/github/PedroBSB/BDATA412/blob/main/07_mds_autoencoders.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/></a></p>
</div>

<div style="background: linear-gradient(to right, #FFF8E1, #FFFEF7); padding: 20px 25px; border-radius: 10px; border: 2px solid #E8B923; margin: 30px 0;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0; font-size: 1.3em;">Student Information</h3>
  <div style="background-color: white; padding: 15px; border-radius: 5px; margin-top: 15px;">
    <p style="margin: 10px 0; color: #333; font-size: 1.05em;"><strong style="color: #4B2E83;">Name:</strong> <span style="color: #666;">[Enter your name]</span></p>
    <p style="margin: 10px 0; color: #333; font-size: 1.05em;"><strong style="color: #4B2E83;">Student ID:</strong> <span style="color: #666;">[Enter your student ID]</span></p>
    <p style="margin: 10px 0; color: #333; font-size: 1.05em;"><strong style="color: #4B2E83;">Date:</strong> <span style="color: #666;">[Enter date]</span></p>
  </div>
</div>

<div style="background: linear-gradient(to right, #F7F4FB, #FFFFFF); padding: 25px; border-radius: 10px; border: 2px solid #E8E0F5; margin: 25px 0;">
  <h2 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; border-bottom: 3px solid #4B2E83; padding-bottom: 10px;">Table of Contents</h2>
  <ul style="font-size: 1.05em; line-height: 2; color: #333; list-style-type: none; padding-left: 0;"><li><strong>Part 1: Classical MDS</strong><ul style='list-style-type: disc; margin-left: 30px;'><li>From Distances to a Map</li><li>Double-Centering &amp; Gram Matrix B</li><li>Eigendecomposition → Coordinates</li><li>Stress &amp; Goodness-of-Fit</li></ul></li><li><strong>Part 2: Metric vs Non-Metric MDS</strong><ul style='list-style-type: disc; margin-left: 30px;'><li>Kruskal Stress</li><li>When MDS = PCA</li><li>Why Axes Are Unlabeled</li><li>Negative Eigenvalues</li></ul></li><li><strong>Part 3: Autoencoders</strong><ul style='list-style-type: disc; margin-left: 30px;'><li>Encoder/Decoder Architecture</li><li>Bottleneck Size k</li><li>Reconstruction Loss ‖x − x̂‖²</li><li>Forward Pass &amp; SGD Weight Update</li></ul></li><li><strong>Part 4: Linear AE ≡ PCA</strong><ul style='list-style-type: disc; margin-left: 30px;'><li>Proof: Optimal Linear AE Recovers PCA Subspace</li><li>Why Nonlinear AE Beats PCA</li></ul></li></ul>
</div>

<div style="background: linear-gradient(to right, #F7F4FB, #FFFFFF); padding: 25px; border-radius: 10px; border: 2px solid #E8E0F5; margin: 25px 0;">
  <h2 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; border-bottom: 3px solid #4B2E83; padding-bottom: 10px;">Learning Objectives</h2>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7; margin-bottom: 10px;">By the end of this section you will be able to:</p>
  <ul style="font-size: 1.05em; color: #333; line-height: 2;"><li>Derive Classical MDS from a distance matrix using double-centering and eigendecomposition</li><li>Compute stress to evaluate MDS embedding quality</li><li>Distinguish metric vs non-metric MDS and explain when MDS reduces to PCA</li><li>Describe the encoder–decoder architecture with a bottleneck layer</li><li>Implement a forward pass and one gradient-descent step for a simple autoencoder</li><li>Prove that an optimal linear autoencoder recovers the PCA subspace</li><li>Explain why nonlinear autoencoders can capture structure that PCA cannot</li><li>Visualize MDS embeddings, reconstruction error, and bottleneck representations using plotly</li></ul>
</div>

In [ ]:
# ── Setup ──
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
np.random.seed(42)
print("Libraries loaded successfully!")

<div style="background: linear-gradient(135deg, #4B2E83 0%, #32006E 100%); padding: 30px; border-radius: 10px; margin: 40px 0 30px 0; box-shadow: 0 5px 15px rgba(75,46,131,0.2);">
  <h1 style="color: #FFF; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; margin: 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">Part 1: Classical MDS</h1>
  <p style="color: #E8D3FF; font-family: 'Encode Sans', 'Arial', sans-serif; font-size: 1.3em; margin: 10px 0; font-weight: 300;">From distances to a map</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Core Idea: From Distances to a Map</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Classical MDS starts with <strong>only pairwise distances</strong> (no coordinates!) and recovers a set of coordinates that reproduce those distances as faithfully as possible. Think: given a table of driving distances between cities, can you draw a map?</p>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;"><strong>Algorithm:</strong> (1) Square the distance matrix: D². (2) Double-center to get the Gram matrix B = -½JD²J where J = I - (1/n)11ᵀ. (3) Eigendecompose B = VΛVᵀ. (4) Take top k eigenvalues/vectors → coordinates X = V_k Λ_k^{1/2}.</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Double-Centering &amp; Gram Matrix B</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">The centering matrix <strong>J = I - (1/n)11ᵀ</strong> removes the mean from each row and column simultaneously. Applying it to -½D² converts squared distances into inner products: <strong>B = XXᵀ</strong> (the Gram matrix). This works because d²(i,j) = b_ii + b_jj - 2b_ij.</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Eigendecomposition → Coordinates</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Since B = XXᵀ is positive semi-definite (for Euclidean distances), we eigendecompose B = VΛVᵀ. The embedding coordinates are <strong>X = V_k Λ_k^{1/2}</strong>, where we keep the top k eigenvalues. Choosing k = 2 gives a 2D map.</p>
</div>

In [ ]:
# Classical MDS from scratch
# City driving distances (in miles) between 6 US cities
cities = ['Seattle', 'Portland', 'San Francisco', 'Los Angeles', 'Denver', 'Phoenix']
D = np.array([
    [   0,  174,  808, 1135,  1321, 1418],
    [ 174,    0,  636,  963,  1238, 1338],
    [ 808,  636,    0,  381,  1235,  753],
    [1135,  963,  381,    0,  1020,  372],
    [1321, 1238, 1235, 1020,     0,  602],
    [1418, 1338,  753,  372,   602,    0]
], dtype=float)

n = len(D)
print(f"Distance matrix D ({n}x{n}):")
print(pd.DataFrame(D, index=cities, columns=cities))

In [ ]:
# Step 1: Square the distance matrix
D2 = D ** 2
print("D\u00b2 (squared distances):")
print(pd.DataFrame(D2.astype(int), index=cities, columns=cities))

In [ ]:
# Step 2: Double-center to get Gram matrix B
J = np.eye(n) - np.ones((n, n)) / n          # centering matrix
B = -0.5 * J @ D2 @ J                         # Gram matrix
print("Gram matrix B = -1/2 * J * D\u00b2 * J:")
print(pd.DataFrame(np.round(B, 1), index=cities, columns=cities))

In [ ]:
# Step 3: Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(B)

# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues of B (sorted descending):")
for i, ev in enumerate(eigenvalues):
    pct = max(0, ev) / np.sum(eigenvalues[eigenvalues > 0]) * 100
    print(f"  \u03bb_{i+1} = {ev:>12.1f}   ({pct:5.1f}% of positive variance)")

In [ ]:
# Step 4: Extract 2D coordinates
k = 2
L_k = np.diag(np.sqrt(np.maximum(eigenvalues[:k], 0)))   # \Lambda_k^{1/2}
V_k = eigenvectors[:, :k]                                 # top-k eigenvectors
X_mds = V_k @ L_k                                         # coordinates

print("MDS 2D coordinates:")
coords_df = pd.DataFrame(X_mds, index=cities, columns=['Dim 1', 'Dim 2'])
print(coords_df.round(1))

In [ ]:
# Visualize the MDS city map
fig = go.Figure(go.Scatter(
    x=X_mds[:, 0], y=X_mds[:, 1], mode='markers+text',
    text=cities, textposition='top center',
    marker=dict(size=14, color='#4B2E83', line=dict(width=2, color='white')),
    textfont=dict(size=12, family='Encode Sans, Arial, sans-serif')))
fig.update_layout(
    title='<b>Classical MDS</b> \u2014 City Map Recovered from Distances Only',
    xaxis_title='MDS Dimension 1',
    yaxis_title='MDS Dimension 2',
    template='plotly_white', height=500, width=650,
    xaxis=dict(scaleanchor='y'),
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Stress &amp; Goodness-of-Fit</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;"><strong>Stress</strong> measures how well the embedding distances match the original distances. <strong>Raw stress = Σ(d_ij - d̂_ij)²</strong>. <strong>Normalized stress-1</strong> (Kruskal) = √[Σ(d_ij - d̂_ij)² / Σ d_ij²]. Values below 0.05 = excellent; 0.1 = fair; above 0.2 = poor.</p>
</div>

In [ ]:
# Compute stress for different embedding dimensions
from scipy.spatial.distance import squareform as sf

def compute_stress(D_orig, X_embed):
    """Compute normalized stress-1 (Kruskal)."""
    D_embed = squareform(pdist(X_embed))
    mask = np.triu_indices_from(D_orig, k=1)
    d_orig = D_orig[mask]
    d_embed = D_embed[mask]
    return np.sqrt(np.sum((d_orig - d_embed)**2) / np.sum(d_orig**2))

stress_by_dim = []
for k in range(1, n):
    L_k = np.diag(np.sqrt(np.maximum(eigenvalues[:k], 0)))
    V_k = eigenvectors[:, :k]
    X_k = V_k @ L_k
    s = compute_stress(D, X_k)
    stress_by_dim.append(s)
    print(f"k={k}: stress = {s:.4f}")

fig = go.Figure(go.Scatter(
    x=list(range(1, n)), y=stress_by_dim, mode='lines+markers',
    marker=dict(size=10, color='#4B2E83'),
    line=dict(color='#4B2E83', width=3)))
fig.add_hline(y=0.05, line_dash='dash', line_color='#388E3C',
              annotation_text='Excellent (< 0.05)')
fig.add_hline(y=0.1, line_dash='dash', line_color='#F57C00',
              annotation_text='Fair (< 0.10)')
fig.update_layout(
    title='<b>Stress vs Embedding Dimension</b>',
    xaxis_title='Embedding Dimension k',
    yaxis_title='Normalized Stress-1',
    template='plotly_white', height=400,
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 1.1: MDS on European Cities</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Given the distance matrix below for 5 European cities, apply classical MDS from scratch to produce 2D coordinates. Compute the stress and plot the result.</p>
</div>

In [ ]:
# Exercise 1.1: European cities MDS
euro_cities = ['London', 'Paris', 'Rome', 'Berlin', 'Madrid']
D_euro = np.array([
    [   0,  340,  1430,  930, 1260],
    [ 340,    0,  1100,  880, 1050],
    [1430, 1100,     0, 1180, 1950],
    [ 930,  880,  1180,    0, 1870],
    [1260, 1050,  1950, 1870,    0]
], dtype=float)

# YOUR CODE HERE
n_e = len(D_euro)
D2_e = D_euro ** 2
J_e = np.eye(n_e) - np.ones((n_e, n_e)) / n_e
B_e = -0.5 * J_e @ D2_e @ J_e

evals_e, evecs_e = np.linalg.eigh(B_e)
idx_e = np.argsort(evals_e)[::-1]
evals_e = evals_e[idx_e]
evecs_e = evecs_e[:, idx_e]

L2_e = np.diag(np.sqrt(np.maximum(evals_e[:2], 0)))
X_euro = evecs_e[:, :2] @ L2_e

stress_e = compute_stress(D_euro, X_euro)
print(f"Stress (2D): {stress_e:.4f}")
print("\nCoordinates:")
print(pd.DataFrame(X_euro, index=euro_cities, columns=['Dim 1', 'Dim 2']).round(1))

fig = go.Figure(go.Scatter(
    x=X_euro[:, 0], y=X_euro[:, 1], mode='markers+text',
    text=euro_cities, textposition='top center',
    marker=dict(size=14, color='#E8B923', line=dict(width=2, color='#4B2E83')),
    textfont=dict(size=12)))
fig.update_layout(
    title=f'<b>Exercise 1.1</b> \u2014 European Cities MDS (stress={stress_e:.4f})',
    xaxis_title='Dim 1', yaxis_title='Dim 2',
    template='plotly_white', height=450, xaxis=dict(scaleanchor='y'),
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 1.2: Verify the Reconstruction</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Using the US cities MDS embedding from above, compute the pairwise distances in the embedding and compare them to the original distances. What is the maximum absolute error? Which city pair has the worst fit?</p>
</div>

In [ ]:
# Exercise 1.2: Reconstruction verification
D_reconstructed = squareform(pdist(X_mds))
error = np.abs(D - D_reconstructed)

print("Original distances:")
print(pd.DataFrame(D.astype(int), index=cities, columns=cities))
print("\nReconstructed distances (2D):")
print(pd.DataFrame(np.round(D_reconstructed, 0).astype(int), index=cities, columns=cities))
print("\nAbsolute error:")
print(pd.DataFrame(np.round(error, 1), index=cities, columns=cities))

mask = np.triu_indices_from(error, k=1)
worst = np.argmax(error[mask])
i_w, j_w = mask[0][worst], mask[1][worst]
print(f"\nMax error: {error[i_w, j_w]:.1f} miles between {cities[i_w]} and {cities[j_w]}")

<div style="background: linear-gradient(135deg, #4B2E83 0%, #32006E 100%); padding: 30px; border-radius: 10px; margin: 40px 0 30px 0; box-shadow: 0 5px 15px rgba(75,46,131,0.2);">
  <h1 style="color: #FFF; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; margin: 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">Part 2: Metric vs Non-Metric MDS</h1>
  <p style="color: #E8D3FF; font-family: 'Encode Sans', 'Arial', sans-serif; font-size: 1.3em; margin: 10px 0; font-weight: 300;">Preserving distances vs preserving rank order</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Kruskal Stress</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;"><strong>Metric MDS</strong> minimizes raw stress: Σ(d_ij - d̂_ij)². <strong>Non-metric MDS</strong> only requires that the <em>rank order</em> of distances is preserved (monotone regression). Kruskal's stress-1 normalizes: <strong>σ₁ = √[Σ(d_ij - d̂_ij)² / Σ d_ij²]</strong>. Non-metric MDS is more flexible but loses magnitude information.</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">When MDS = PCA</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">If the input distances are <strong>Euclidean distances computed from centered data</strong>, then classical MDS produces <em>exactly</em> the same embedding as PCA (up to rotation/reflection). Reason: the Gram matrix B = XXᵀ has the same eigenstructure as the covariance matrix XᵀX (just scaled). The eigenvalues of B are n times the eigenvalues of the covariance.</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Why Axes Are Unlabeled</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Unlike PCA where each axis corresponds to a principal component with loadings on original features, MDS axes have <strong>no intrinsic meaning</strong>. The embedding is defined only up to rotation, reflection, and translation. "Dimension 1" is just the direction of largest spread — it has no feature interpretation.</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Negative Eigenvalues</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">If the input distances are <strong>not Euclidean</strong> (e.g., geodesic, survey dissimilarities), the Gram matrix B may have negative eigenvalues. This means no Euclidean embedding perfectly reproduces those distances. Solutions: (a) ignore negative eigenvalues (set to 0), (b) add a constant to all off-diagonal distances, (c) use non-metric MDS instead.</p>
</div>

In [ ]:
# Demonstrate MDS = PCA when distances are Euclidean
np.random.seed(42)
X_data = np.random.randn(50, 5)         # 50 points in 5D
X_centered = X_data - X_data.mean(axis=0)

# PCA embedding
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_centered)

# Classical MDS on Euclidean distances
D_euc = squareform(pdist(X_centered))
n_pts = len(D_euc)
J_pts = np.eye(n_pts) - np.ones((n_pts, n_pts)) / n_pts
B_pts = -0.5 * J_pts @ (D_euc**2) @ J_pts
evals_pts, evecs_pts = np.linalg.eigh(B_pts)
idx_pts = np.argsort(evals_pts)[::-1]
evals_pts = evals_pts[idx_pts]
evecs_pts = evecs_pts[:, idx_pts]
X_mds_pts = evecs_pts[:, :2] @ np.diag(np.sqrt(np.maximum(evals_pts[:2], 0)))

# Align MDS to PCA (Procrustes-like sign flip)
for col in range(2):
    if np.corrcoef(X_pca[:, col], X_mds_pts[:, col])[0, 1] < 0:
        X_mds_pts[:, col] *= -1

fig = make_subplots(rows=1, cols=2, subplot_titles=['PCA Embedding', 'Classical MDS Embedding'])
fig.add_trace(go.Scatter(x=X_pca[:, 0], y=X_pca[:, 1], mode='markers',
    marker=dict(size=7, color='#4B2E83'), name='PCA'), row=1, col=1)
fig.add_trace(go.Scatter(x=X_mds_pts[:, 0], y=X_mds_pts[:, 1], mode='markers',
    marker=dict(size=7, color='#E8B923'), name='MDS'), row=1, col=2)
fig.update_layout(template='plotly_white', height=400, showlegend=False,
    title='<b>MDS \u2261 PCA</b> when distances are Euclidean (centered data)',
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

max_diff = np.max(np.abs(X_pca - X_mds_pts))
print(f"Max absolute difference between PCA and MDS coordinates: {max_diff:.10f}")

In [ ]:
# sklearn metric vs non-metric MDS on city distances
mds_metric = MDS(n_components=2, dissimilarity='precomputed', random_state=42,
                 normalized_stress='auto')
X_metric = mds_metric.fit_transform(D)

mds_nonmetric = MDS(n_components=2, dissimilarity='precomputed', random_state=42,
                    metric=False, normalized_stress='auto')
X_nonmetric = mds_nonmetric.fit_transform(D)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Metric MDS', 'Non-Metric MDS'])
for i, (X_emb, col) in enumerate([(X_metric, 1), (X_nonmetric, 2)]):
    fig.add_trace(go.Scatter(
        x=X_emb[:, 0], y=X_emb[:, 1], mode='markers+text',
        text=cities, textposition='top center',
        marker=dict(size=12, color=['#4B2E83','#E8B923','#22D3A5','#E8608A','#4F8EF7','#F57C00']),
        textfont=dict(size=11), showlegend=False), row=1, col=col)
fig.update_layout(template='plotly_white', height=420,
    title='<b>Metric vs Non-Metric MDS</b> \u2014 US Cities',
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

print(f"Metric MDS stress:     {mds_metric.stress_:.2f}")
print(f"Non-Metric MDS stress: {mds_nonmetric.stress_:.2f}")

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 2.1: Negative Eigenvalues</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Create a non-Euclidean distance matrix (e.g., triangle inequality violation) and run classical MDS. Verify that the Gram matrix B has at least one negative eigenvalue. What happens when you embed into 2D?</p>
</div>

In [ ]:
# Exercise 2.1: Non-Euclidean distances
D_non_euc = np.array([
    [0,  1,  1,  5],
    [1,  0,  5,  1],
    [1,  5,  0,  1],
    [5,  1,  1,  0]
], dtype=float)

n_ne = len(D_non_euc)
J_ne = np.eye(n_ne) - np.ones((n_ne, n_ne)) / n_ne
B_ne = -0.5 * J_ne @ (D_non_euc**2) @ J_ne

evals_ne, evecs_ne = np.linalg.eigh(B_ne)
idx_ne = np.argsort(evals_ne)[::-1]
evals_ne = evals_ne[idx_ne]
evecs_ne = evecs_ne[:, idx_ne]

print("Eigenvalues of B:")
for i, ev in enumerate(evals_ne):
    flag = " <-- NEGATIVE" if ev < -1e-10 else ""
    print(f"  \u03bb_{i+1} = {ev:.4f}{flag}")

# Embed anyway (set negative eigenvalues to 0)
X_ne = evecs_ne[:, :2] @ np.diag(np.sqrt(np.maximum(evals_ne[:2], 0)))
labels_ne = ['A', 'B', 'C', 'D']

fig = go.Figure(go.Scatter(
    x=X_ne[:, 0], y=X_ne[:, 1], mode='markers+text',
    text=labels_ne, textposition='top center',
    marker=dict(size=14, color='#C62828', line=dict(width=2, color='white'))))
fig.update_layout(title='<b>Exercise 2.1</b> \u2014 MDS with Non-Euclidean Distances',
    template='plotly_white', height=400,
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()
print("\nNote: Negative eigenvalues indicate the distances cannot be perfectly embedded in Euclidean space.")

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 2.2: Compare Metric and Non-Metric</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Generate 30 random points in 10D. Compute their Euclidean distances. Run both metric and non-metric MDS (sklearn) with n_components=2. Compare the Shepard diagrams (original distance vs embedded distance).</p>
</div>

In [ ]:
# Exercise 2.2: Shepard diagrams
np.random.seed(42)
X_10d = np.random.randn(30, 10)
D_10d = squareform(pdist(X_10d))

mds_m = MDS(n_components=2, dissimilarity='precomputed', random_state=42,
            normalized_stress='auto')
mds_nm = MDS(n_components=2, dissimilarity='precomputed', random_state=42,
             metric=False, normalized_stress='auto')
X_m = mds_m.fit_transform(D_10d)
X_nm = mds_nm.fit_transform(D_10d)

D_m = squareform(pdist(X_m))
D_nm = squareform(pdist(X_nm))
mask_upper = np.triu_indices_from(D_10d, k=1)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Metric MDS \u2014 Shepard Diagram', 'Non-Metric MDS \u2014 Shepard Diagram'])
fig.add_trace(go.Scatter(x=D_10d[mask_upper], y=D_m[mask_upper], mode='markers',
    marker=dict(size=4, color='#4B2E83', opacity=0.6), name='Metric'), row=1, col=1)
fig.add_trace(go.Scatter(x=D_10d[mask_upper], y=D_nm[mask_upper], mode='markers',
    marker=dict(size=4, color='#E8B923', opacity=0.6), name='Non-Metric'), row=1, col=2)
# Add perfect fit line
d_range = [D_10d[mask_upper].min(), D_10d[mask_upper].max()]
for col in [1, 2]:
    fig.add_trace(go.Scatter(x=d_range, y=d_range, mode='lines',
        line=dict(color='red', dash='dash', width=1), showlegend=False), row=1, col=col)
fig.update_xaxes(title_text='Original Distance', row=1, col=1)
fig.update_xaxes(title_text='Original Distance', row=1, col=2)
fig.update_yaxes(title_text='Embedded Distance', row=1, col=1)
fig.update_yaxes(title_text='Embedded Distance', row=1, col=2)
fig.update_layout(template='plotly_white', height=420, showlegend=False,
    title='<b>Shepard Diagrams</b> \u2014 How well are distances preserved?',
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

<div style="background: linear-gradient(135deg, #4B2E83 0%, #32006E 100%); padding: 30px; border-radius: 10px; margin: 40px 0 30px 0; box-shadow: 0 5px 15px rgba(75,46,131,0.2);">
  <h1 style="color: #FFF; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; margin: 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">Part 3: Autoencoders</h1>
  <p style="color: #E8D3FF; font-family: 'Encode Sans', 'Arial', sans-serif; font-size: 1.3em; margin: 10px 0; font-weight: 300;">Learning compressed representations via reconstruction</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Encoder/Decoder Architecture</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">An autoencoder has two halves: <strong>Encoder</strong> f: ℝᵈ → ℝᵏ maps high-dimensional input x to a low-dimensional code z = f(x). <strong>Decoder</strong> g: ℝᵏ → ℝᵈ reconstructs x̂ = g(z). The network is trained to minimize <strong>reconstruction loss</strong> ‖x - x̂‖².</p>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">The simplest linear autoencoder: z = W_e x (encode), x̂ = W_d z (decode). With nonlinear activations: z = σ(W_e x + b_e), x̂ = W_d z + b_d.</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Bottleneck Size k</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">The bottleneck (code) dimension k forces the network to learn a compressed representation. If k < d, the autoencoder must discard information. <strong>k too small</strong>: underfitting, high reconstruction error. <strong>k too large</strong>: trivial identity mapping, no compression. The sweet spot captures the intrinsic dimensionality of the data.</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Reconstruction Loss ‖x − x̂‖²</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">The loss for a single sample: <strong>L = ‖x - x̂‖² = Σ(x_i - x̂_i)²</strong>. Over a dataset of N samples: <strong>L = (1/N) Σ ‖x⁽ʲ⁾ - x̂⁽ʲ⁾‖²</strong>. This is the mean squared error (MSE). Training minimizes L with respect to encoder and decoder weights.</p>
</div>

In [ ]:
# Simple autoencoder in numpy: forward pass
np.random.seed(42)

# Architecture: 5D -> 2D -> 5D (linear autoencoder)
d_in = 5
k = 2  # bottleneck dimension

# Initialize weights
W_enc = np.random.randn(k, d_in) * 0.5    # encoder: (k x d)
b_enc = np.zeros(k)                         # encoder bias
W_dec = np.random.randn(d_in, k) * 0.5    # decoder: (d x k)
b_dec = np.zeros(d_in)                      # decoder bias

# Sample input
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

# Forward pass (linear)
z = W_enc @ x + b_enc        # encode: R^5 -> R^2
x_hat = W_dec @ z + b_dec    # decode: R^2 -> R^5

# Reconstruction loss
loss = np.sum((x - x_hat)**2)

print(f"Input x:       {x}")
print(f"Code z:        {np.round(z, 4)}")
print(f"Recon x\u0302:      {np.round(x_hat, 4)}")
print(f"Loss \u2016x-x\u0302\u2016\u00b2:  {loss:.4f}")

In [ ]:
# Visualize encoder bottleneck
fig = go.Figure()

# Draw network architecture
layers = [d_in, k, d_in]
layer_names = ['Input (d=5)', 'Bottleneck (k=2)', 'Output (d=5)']
layer_x = [0, 1, 2]
colors = ['#4B2E83', '#E8B923', '#4B2E83']

for l, (n_nodes, lx, col) in enumerate(zip(layers, layer_x, colors)):
    y_positions = np.linspace(-n_nodes/2 + 0.5, n_nodes/2 - 0.5, n_nodes)
    # Draw nodes
    fig.add_trace(go.Scatter(
        x=[lx]*n_nodes, y=y_positions, mode='markers+text',
        marker=dict(size=30, color=col, line=dict(width=2, color='white')),
        text=[f'{v:.1f}' for v in (x if l==0 else (z if l==1 else x_hat))],
        textfont=dict(color='white', size=9),
        showlegend=False, hoverinfo='skip'))
    
    # Draw connections to next layer
    if l < len(layers) - 1:
        n_next = layers[l+1]
        y_next = np.linspace(-n_next/2 + 0.5, n_next/2 - 0.5, n_next)
        for yi in y_positions:
            for yj in y_next:
                fig.add_shape(type='line', x0=lx, y0=yi, x1=layer_x[l+1], y1=yj,
                    line=dict(color='rgba(75,46,131,0.15)', width=1))

# Layer labels
for lx, name in zip(layer_x, layer_names):
    fig.add_annotation(x=lx, y=-3.5, text=f'<b>{name}</b>',
        showarrow=False, font=dict(size=11, color='#333'))

fig.add_annotation(x=0.5, y=2.5, text='<b>Encoder</b> (W_enc)',
    showarrow=False, font=dict(size=12, color='#4B2E83'))
fig.add_annotation(x=1.5, y=2.5, text='<b>Decoder</b> (W_dec)',
    showarrow=False, font=dict(size=12, color='#4B2E83'))

fig.update_layout(
    title='<b>Autoencoder Architecture</b> \u2014 5D \u2192 2D \u2192 5D',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 2.5]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-4.5, 3.5]),
    template='plotly_white', height=450, width=600,
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Forward Pass &amp; SGD Weight Update</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;"><strong>Forward pass:</strong> z = W_e x, x̂ = W_d z. <strong>Loss:</strong> L = ‖x - x̂‖². <strong>Backward pass (gradients):</strong> ∂L/∂W_d = -2(x - x̂)zᵀ, ∂L/∂W_e = -2 W_dᵀ(x - x̂)xᵀ. <strong>Update:</strong> W ← W - η · ∂L/∂W.</p>
</div>

In [ ]:
# Train a simple linear autoencoder with gradient descent
np.random.seed(42)

# Generate data: 100 points in 5D with intrinsic dimension ~2
t = np.random.randn(100, 2)
A = np.random.randn(5, 2)           # true mixing matrix
X_train = t @ A.T + 0.1 * np.random.randn(100, 5)   # 5D data from 2D sources
X_train = X_train - X_train.mean(axis=0)              # center

# Initialize weights
d, k_ae = 5, 2
W_e = np.random.randn(k_ae, d) * 0.1
W_d = np.random.randn(d, k_ae) * 0.1

lr = 0.0001
losses = []

for epoch in range(200):
    epoch_loss = 0
    for i in range(len(X_train)):
        x_i = X_train[i]
        
        # Forward pass
        z_i = W_e @ x_i           # encode
        x_hat_i = W_d @ z_i       # decode
        
        # Loss
        residual = x_i - x_hat_i
        loss_i = np.sum(residual**2)
        epoch_loss += loss_i
        
        # Gradients (chain rule)
        dL_dW_d = -2 * np.outer(residual, z_i)       # (d x k)
        dL_dW_e = -2 * np.outer(W_d.T @ residual, x_i)  # (k x d)
        
        # SGD update
        W_d -= lr * dL_dW_d
        W_e -= lr * dL_dW_e
    
    losses.append(epoch_loss / len(X_train))
    if epoch % 40 == 0:
        print(f"Epoch {epoch:>3d}: MSE = {losses[-1]:.4f}")

print(f"Epoch {199:>3d}: MSE = {losses[-1]:.4f}")

fig = go.Figure(go.Scatter(x=list(range(200)), y=losses, mode='lines',
    line=dict(color='#4B2E83', width=2.5)))
fig.update_layout(
    title='<b>Autoencoder Training</b> \u2014 Reconstruction Loss over Epochs',
    xaxis_title='Epoch', yaxis_title='MSE Loss',
    template='plotly_white', height=380,
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

In [ ]:
# Visualize bottleneck codes and reconstructions
Z_codes = (W_e @ X_train.T).T   # all codes
X_recon = (W_d @ Z_codes.T).T   # all reconstructions

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Bottleneck Codes (2D)', 'Reconstruction: Original vs Decoded (Dim 1 vs 2)'])

fig.add_trace(go.Scatter(x=Z_codes[:, 0], y=Z_codes[:, 1], mode='markers',
    marker=dict(size=5, color='#E8B923', opacity=0.7), name='Codes'), row=1, col=1)

fig.add_trace(go.Scatter(x=X_train[:, 0], y=X_train[:, 1], mode='markers',
    marker=dict(size=5, color='#4B2E83', opacity=0.6), name='Original'), row=1, col=2)
fig.add_trace(go.Scatter(x=X_recon[:, 0], y=X_recon[:, 1], mode='markers',
    marker=dict(size=5, color='#E8608A', opacity=0.6, symbol='x'), name='Reconstructed'), row=1, col=2)

fig.update_layout(template='plotly_white', height=400,
    title='<b>Autoencoder Bottleneck &amp; Reconstruction</b>',
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 3.1: Vary the Bottleneck Size</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Train separate linear autoencoders with bottleneck sizes k = 1, 2, 3, 4 on the same 5D data. Plot the final reconstruction loss for each k. At which k does the loss plateau? Why?</p>
</div>

In [ ]:
# Exercise 3.1: Bottleneck size comparison
final_losses = []
for k_test in [1, 2, 3, 4]:
    np.random.seed(42)
    We = np.random.randn(k_test, d) * 0.1
    Wd = np.random.randn(d, k_test) * 0.1
    
    for epoch in range(200):
        for i in range(len(X_train)):
            xi = X_train[i]
            zi = We @ xi
            xh = Wd @ zi
            res = xi - xh
            Wd -= lr * (-2 * np.outer(res, zi))
            We -= lr * (-2 * np.outer(Wd.T @ res, xi))
    
    # Final loss
    Z_test = (We @ X_train.T).T
    X_test_recon = (Wd @ Z_test.T).T
    mse = np.mean(np.sum((X_train - X_test_recon)**2, axis=1))
    final_losses.append(mse)
    print(f"k={k_test}: final MSE = {mse:.4f}")

fig = go.Figure(go.Bar(x=[f'k={k}' for k in [1,2,3,4]], y=final_losses,
    marker_color=['#E8608A','#4B2E83','#22D3A5','#E8B923'],
    text=[f'{l:.3f}' for l in final_losses], textposition='outside'))
fig.update_layout(
    title='<b>Exercise 3.1</b> \u2014 Reconstruction Loss vs Bottleneck Size',
    xaxis_title='Bottleneck Size k', yaxis_title='MSE',
    template='plotly_white', height=400,
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()
print("\nLoss plateaus at k=2 because the data has intrinsic dimension \u2248 2.")

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 3.2: One Gradient Step by Hand</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Given W_e = [[1, 0], [0, 1]], W_d = [[1, 0], [0, 1]], and x = [3, 4], compute by hand: (1) z, (2) x̂, (3) loss, (4) gradients ∂L/∂W_d and ∂L/∂W_e, (5) updated weights with η = 0.01. Verify with numpy.</p>
</div>

In [ ]:
# Exercise 3.2: Manual gradient step
We_ex = np.array([[1.0, 0.0], [0.0, 1.0]])
Wd_ex = np.array([[1.0, 0.0], [0.0, 1.0]])
x_ex = np.array([3.0, 4.0])
eta = 0.01

# Forward
z_ex = We_ex @ x_ex
xhat_ex = Wd_ex @ z_ex
loss_ex = np.sum((x_ex - xhat_ex)**2)

print(f"z = W_e @ x = {z_ex}")
print(f"x\u0302 = W_d @ z = {xhat_ex}")
print(f"Loss = \u2016x - x\u0302\u2016\u00b2 = {loss_ex}")
print(f"\nSince x\u0302 = x (identity), loss = 0 and gradients = 0.")
print(f"Weights remain unchanged. The identity is already optimal for k=d.")

# Now with a non-identity encoder
We_ex2 = np.array([[0.5, 0.3], [0.2, 0.8]])
Wd_ex2 = np.array([[0.7, 0.1], [0.4, 0.6]])

z_ex2 = We_ex2 @ x_ex
xhat_ex2 = Wd_ex2 @ z_ex2
res_ex2 = x_ex - xhat_ex2
loss_ex2 = np.sum(res_ex2**2)

dL_Wd = -2 * np.outer(res_ex2, z_ex2)
dL_We = -2 * np.outer(Wd_ex2.T @ res_ex2, x_ex)

print(f"\n--- Non-identity case ---")
print(f"z = {np.round(z_ex2, 4)}")
print(f"x\u0302 = {np.round(xhat_ex2, 4)}")
print(f"residual = {np.round(res_ex2, 4)}")
print(f"Loss = {loss_ex2:.4f}")
print(f"\n\u2202L/\u2202W_d =\n{np.round(dL_Wd, 4)}")
print(f"\u2202L/\u2202W_e =\n{np.round(dL_We, 4)}")

We_new = We_ex2 - eta * dL_We
Wd_new = Wd_ex2 - eta * dL_Wd
print(f"\nUpdated W_e =\n{np.round(We_new, 4)}")
print(f"Updated W_d =\n{np.round(Wd_new, 4)}")

<div style="background: linear-gradient(135deg, #4B2E83 0%, #32006E 100%); padding: 30px; border-radius: 10px; margin: 40px 0 30px 0; box-shadow: 0 5px 15px rgba(75,46,131,0.2);">
  <h1 style="color: #FFF; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; margin: 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">Part 4: Linear AE ≡ PCA</h1>
  <p style="color: #E8D3FF; font-family: 'Encode Sans', 'Arial', sans-serif; font-size: 1.3em; margin: 10px 0; font-weight: 300;">The optimal linear autoencoder recovers PCA</p>
</div>

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Proof: Optimal Linear AE Recovers PCA Subspace</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">A linear autoencoder minimizes L = (1/N)Σ‖x⁽ʲ⁾ - W_d W_e x⁽ʲ⁾‖². The product P = W_d W_e is a rank-k matrix. The Eckart–Young theorem says the rank-k matrix that best approximates X (in Frobenius norm) is the projection onto the top-k principal components. Therefore, at the optimum, <strong>W_d W_e = V_k V_kᵀ</strong> where V_k are the top-k eigenvectors of the covariance matrix.</p>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">The individual W_e and W_d are <em>not</em> unique — any invertible k×k matrix R gives another optimum: W_e' = R W_e, W_d' = W_d R⁻¹. But the <strong>subspace</strong> they span is always the PCA subspace.</p>
</div>

In [ ]:
# Demonstrate: trained linear AE vs PCA
# The AE reconstruction should match PCA reconstruction

# PCA reconstruction
pca_2d = PCA(n_components=2)
Z_pca = pca_2d.fit_transform(X_train)
X_pca_recon = pca_2d.inverse_transform(Z_pca)

# AE reconstruction (from trained weights above)
Z_ae = (W_e @ X_train.T).T
X_ae_recon = (W_d @ Z_ae.T).T

mse_pca = np.mean(np.sum((X_train - X_pca_recon)**2, axis=1))
mse_ae = np.mean(np.sum((X_train - X_ae_recon)**2, axis=1))

print(f"PCA reconstruction MSE:          {mse_pca:.6f}")
print(f"Linear AE reconstruction MSE:    {mse_ae:.6f}")
print(f"\nThe AE MSE is close to PCA MSE (may not be exact due to SGD convergence).")

# Check subspace alignment
# Project PCA components onto AE subspace
P_ae = W_d @ W_e                     # AE projection matrix
P_pca = pca_2d.components_.T @ pca_2d.components_   # PCA projection matrix

# Subspace angle
_, s_vals, _ = np.linalg.svd(pca_2d.components_ @ W_d)  # (2xd)(dxk)
principal_angles = np.arccos(np.clip(s_vals, -1, 1))
print(f"\nPrincipal angles between AE and PCA subspaces: {np.round(np.degrees(principal_angles), 2)} degrees")

<div style="border-left: 5px solid #4B2E83; background-color: #F9F7FC; padding: 20px 25px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #4B2E83; font-family: 'Encode Sans', 'Arial', sans-serif; margin-top: 0;">Why Nonlinear AE Beats PCA</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">PCA (and linear AE) can only capture <strong>linear</strong> subspaces. If data lies on a curved manifold (e.g., swiss roll, S-curve), PCA flattens the structure. A <strong>nonlinear autoencoder</strong> with activation functions (ReLU, sigmoid) can learn curved mappings, capturing intrinsic structure that PCA misses entirely. Trade-off: more capacity = more risk of overfitting.</p>
</div>

In [ ]:
# Demonstrate nonlinear data: Swiss roll (2D manifold in 3D)
np.random.seed(42)
n_sr = 300
t_sr = 1.5 * np.pi * (1 + 2 * np.random.rand(n_sr))
x_sr = t_sr * np.cos(t_sr)
y_sr = 20 * np.random.rand(n_sr)
z_sr = t_sr * np.sin(t_sr)
X_swiss = np.column_stack([x_sr, y_sr, z_sr])
X_swiss_c = X_swiss - X_swiss.mean(axis=0)

# PCA on swiss roll
pca_sr = PCA(n_components=2)
Z_pca_sr = pca_sr.fit_transform(X_swiss_c)

# MDS on swiss roll
D_sr = squareform(pdist(X_swiss_c))
mds_sr = MDS(n_components=2, dissimilarity='precomputed', random_state=42,
             normalized_stress='auto')
Z_mds_sr = mds_sr.fit_transform(D_sr)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['PCA Embedding (linear)', 'MDS Embedding'])

fig.add_trace(go.Scatter(x=Z_pca_sr[:, 0], y=Z_pca_sr[:, 1], mode='markers',
    marker=dict(size=4, color=t_sr, colorscale='Viridis', showscale=False),
    name='PCA'), row=1, col=1)
fig.add_trace(go.Scatter(x=Z_mds_sr[:, 0], y=Z_mds_sr[:, 1], mode='markers',
    marker=dict(size=4, color=t_sr, colorscale='Viridis', showscale=False),
    name='MDS'), row=1, col=2)

fig.update_layout(template='plotly_white', height=400, showlegend=False,
    title='<b>Swiss Roll</b> \u2014 Linear methods fail on curved manifolds',
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()

print("Both PCA and Euclidean MDS 'fold' the swiss roll instead of unrolling it.")
print("A nonlinear autoencoder could learn to unroll the manifold.")

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 4.1: Compare PCA and Linear AE Subspaces</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Generate 200 points in 4D from a 2D latent space (z @ A.T + noise). Train a linear AE with k=2 for 300 epochs. Compare the projection matrix W_d W_e with V_k V_kᵀ from PCA. Compute the Frobenius norm of the difference.</p>
</div>

In [ ]:
# Exercise 4.1: Subspace comparison
np.random.seed(123)
n_ex = 200
d_ex = 4
k_ex = 2

z_latent = np.random.randn(n_ex, k_ex)
A_mix = np.random.randn(d_ex, k_ex)
X_ex = z_latent @ A_mix.T + 0.05 * np.random.randn(n_ex, d_ex)
X_ex = X_ex - X_ex.mean(axis=0)

# PCA projection matrix
pca_ex = PCA(n_components=k_ex)
pca_ex.fit(X_ex)
P_pca_ex = pca_ex.components_.T @ pca_ex.components_   # d x d

# Train linear AE
We_ex4 = np.random.randn(k_ex, d_ex) * 0.1
Wd_ex4 = np.random.randn(d_ex, k_ex) * 0.1
lr_ex = 0.00005

for epoch in range(300):
    for i in range(n_ex):
        xi = X_ex[i]
        zi = We_ex4 @ xi
        xh = Wd_ex4 @ zi
        res = xi - xh
        Wd_ex4 -= lr_ex * (-2 * np.outer(res, zi))
        We_ex4 -= lr_ex * (-2 * np.outer(Wd_ex4.T @ res, xi))

P_ae_ex = Wd_ex4 @ We_ex4   # d x d
frob_diff = np.linalg.norm(P_pca_ex - P_ae_ex, 'fro')

print(f"PCA projection matrix P_pca:\n{np.round(P_pca_ex, 4)}")
print(f"\nAE projection matrix P_ae = W_d W_e:\n{np.round(P_ae_ex, 4)}")
print(f"\nFrobenius norm \u2016P_pca - P_ae\u2016_F = {frob_diff:.6f}")
print("A small value confirms the linear AE converged to the PCA subspace.")

<div style="background-color: #FFF3E0; border-left: 4px solid #F57C00; padding: 15px 20px; margin: 20px 0; border-radius: 5px;">
  <h3 style="color: #F57C00; margin-top: 0;">Exercise 4.2: Nonlinear AE with ReLU</h3>
  <p style="font-size: 1.05em; color: #555; line-height: 1.7;">Add a ReLU activation to the encoder: z = max(0, W_e x + b_e). Train on the same data and compare reconstruction error with the linear AE. Does the nonlinear AE do better? Why or why not for this dataset?</p>
</div>

In [ ]:
# Exercise 4.2: Nonlinear AE with ReLU
np.random.seed(42)

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

# Architecture: 5D -> 2D (ReLU) -> 5D
We_nl = np.random.randn(k_ae, d) * 0.1
be_nl = np.zeros(k_ae)
Wd_nl = np.random.randn(d, k_ae) * 0.1
bd_nl = np.zeros(d)

losses_nl = []
lr_nl = 0.0001

for epoch in range(200):
    epoch_loss = 0
    for i in range(len(X_train)):
        xi = X_train[i]
        
        # Forward (with ReLU)
        h_pre = We_nl @ xi + be_nl    # pre-activation
        zi = relu(h_pre)               # code
        xh = Wd_nl @ zi + bd_nl        # reconstruction
        
        res = xi - xh
        loss_i = np.sum(res**2)
        epoch_loss += loss_i
        
        # Backward
        dL_dxh = -2 * res
        dL_dWd = np.outer(dL_dxh, zi)
        dL_dbd = dL_dxh
        dL_dzi = Wd_nl.T @ dL_dxh
        dL_dhpre = dL_dzi * relu_grad(h_pre)
        dL_dWe = np.outer(dL_dhpre, xi)
        dL_dbe = dL_dhpre
        
        # Update
        Wd_nl -= lr_nl * dL_dWd
        bd_nl -= lr_nl * dL_dbd
        We_nl -= lr_nl * dL_dWe
        be_nl -= lr_nl * dL_dbe
    
    losses_nl.append(epoch_loss / len(X_train))

print(f"Linear AE final MSE:    {losses[-1]:.4f}")
print(f"Nonlinear AE final MSE: {losses_nl[-1]:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(200)), y=losses, mode='lines',
    name='Linear AE', line=dict(color='#4B2E83', width=2.5)))
fig.add_trace(go.Scatter(x=list(range(200)), y=losses_nl, mode='lines',
    name='Nonlinear AE (ReLU)', line=dict(color='#E8B923', width=2.5)))
fig.update_layout(
    title='<b>Exercise 4.2</b> \u2014 Linear vs Nonlinear Autoencoder',
    xaxis_title='Epoch', yaxis_title='MSE',
    template='plotly_white', height=400,
    font=dict(family='Encode Sans, Arial, sans-serif'))
fig.show()
print("\nFor linearly-generated data, the nonlinear AE does not significantly beat the linear one.")
print("Nonlinear AEs shine when the data lies on a curved manifold.")

<div style="background: linear-gradient(135deg, #4B2E83 0%, #32006E 100%); padding: 30px; border-radius: 10px; margin: 40px 0 30px 0; box-shadow: 0 5px 15px rgba(75,46,131,0.2);">
  <h1 style="color: #FFF; font-family: 'Encode Sans', 'Arial', sans-serif; font-weight: 700; margin: 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">Interactive Dashboard</h1>
  <p style="color: #E8D3FF; font-family: 'Encode Sans', 'Arial', sans-serif; font-size: 1.3em; margin: 10px 0; font-weight: 300;">MDS, Autoencoders &amp; Comparison</p>
</div>

In [ ]:
# ============================================================
#  MDS & AUTOENCODERS DASHBOARD
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
import numpy as np
from scipy.spatial.distance import pdist, squareform

np.random.seed(42)

# \u2500\u2500 Theme \u2500\u2500
C = {
    "bg": "#0F1117", "card": "#171A23", "grid": "#262B3A",
    "text": "#E8EAF0", "muted": "#8B90A0",
    "primary": "#4B2E83", "accent": "#E8B923",
    "green": "#22D3A5", "pink": "#E8608A",
    "palette": ["#4B2E83","#E8B923","#22D3A5","#E8608A","#4F8EF7","#F57C00"]
}
BASE = dict(paper_bgcolor=C["card"], plot_bgcolor=C["card"],
            font=dict(family="Inter, Segoe UI, sans-serif", color=C["text"], size=12),
            margin=dict(l=55, r=30, t=60, b=45))

def style(fig):
    fig.update_xaxes(gridcolor=C["grid"], zerolinecolor=C["grid"], tickfont=dict(color=C["muted"]))
    fig.update_yaxes(gridcolor=C["grid"], zerolinecolor=C["grid"], tickfont=dict(color=C["muted"]))
    return fig

figs = {}

# ================================================================
# TAB 1: MDS
# ================================================================

# City map from distances
cities_d = ['Seattle', 'Portland', 'San Francisco', 'Los Angeles', 'Denver', 'Phoenix']
D_d = np.array([
    [   0,  174,  808, 1135,  1321, 1418],
    [ 174,    0,  636,  963,  1238, 1338],
    [ 808,  636,    0,  381,  1235,  753],
    [1135,  963,  381,    0,  1020,  372],
    [1321, 1238, 1235, 1020,     0,  602],
    [1418, 1338,  753,  372,   602,    0]
], dtype=float)

n_d = len(D_d)
J_d = np.eye(n_d) - np.ones((n_d, n_d)) / n_d
B_d = -0.5 * J_d @ (D_d**2) @ J_d
evals_d, evecs_d = np.linalg.eigh(B_d)
idx_d = np.argsort(evals_d)[::-1]
evals_d = evals_d[idx_d]
evecs_d = evecs_d[:, idx_d]
X_city = evecs_d[:, :2] @ np.diag(np.sqrt(np.maximum(evals_d[:2], 0)))

f = go.Figure(go.Scatter(
    x=X_city[:, 0], y=X_city[:, 1], mode='markers+text',
    text=cities_d, textposition='top center',
    marker=dict(size=14, color=C["palette"], line=dict(width=2, color='white')),
    textfont=dict(size=11, color=C["text"])))
f.update_layout(**BASE, height=450,
    title=dict(text="<b>City Map from Distances Only (Classical MDS)</b>", font=dict(size=16)),
    xaxis=dict(scaleanchor='y', title='MDS Dim 1'),
    yaxis=dict(title='MDS Dim 2'))
style(f)
figs["city_map"] = f

# Stress by dimension
stress_vals = []
for kk in range(1, n_d):
    Lk = np.diag(np.sqrt(np.maximum(evals_d[:kk], 0)))
    Vk = evecs_d[:, :kk]
    Xk = Vk @ Lk
    D_embed = squareform(pdist(Xk))
    mask_s = np.triu_indices_from(D_d, k=1)
    stress_vals.append(np.sqrt(np.sum((D_d[mask_s] - D_embed[mask_s])**2) / np.sum(D_d[mask_s]**2)))

f = go.Figure(go.Scatter(x=list(range(1, n_d)), y=stress_vals, mode='lines+markers',
    marker=dict(size=10, color=C["accent"]), line=dict(color=C["accent"], width=3)))
f.add_hline(y=0.05, line_dash='dash', line_color=C["green"],
    annotation=dict(text='Excellent', font=dict(color=C["green"])))
f.add_hline(y=0.1, line_dash='dash', line_color=C["pink"],
    annotation=dict(text='Fair', font=dict(color=C["pink"])))
f.update_layout(**BASE, height=380,
    title=dict(text="<b>Stress vs Embedding Dimension</b>", font=dict(size=16)),
    xaxis_title='Dimension k', yaxis_title='Normalized Stress-1')
style(f)
figs["stress"] = f

# ================================================================
# TAB 2: AUTOENCODER
# ================================================================

# Reconstruction error by bottleneck size
t_ae = np.random.randn(100, 2)
A_ae = np.random.randn(5, 2)
X_ae_data = t_ae @ A_ae.T + 0.1 * np.random.randn(100, 5)
X_ae_data = X_ae_data - X_ae_data.mean(axis=0)

ae_losses = []
for k_b in [1, 2, 3, 4]:
    np.random.seed(42)
    We_b = np.random.randn(k_b, 5) * 0.1
    Wd_b = np.random.randn(5, k_b) * 0.1
    for ep in range(200):
        for ii in range(len(X_ae_data)):
            xi = X_ae_data[ii]
            zi = We_b @ xi
            xh = Wd_b @ zi
            res = xi - xh
            Wd_b -= 0.0001 * (-2 * np.outer(res, zi))
            We_b -= 0.0001 * (-2 * np.outer(Wd_b.T @ res, xi))
    Z_b = (We_b @ X_ae_data.T).T
    X_b_recon = (Wd_b @ Z_b.T).T
    ae_losses.append(np.mean(np.sum((X_ae_data - X_b_recon)**2, axis=1)))

f = go.Figure(go.Bar(
    x=['k=1', 'k=2', 'k=3', 'k=4'], y=ae_losses,
    marker_color=[C["pink"], C["primary"], C["green"], C["accent"]],
    text=[f'{l:.3f}' for l in ae_losses], textposition='outside',
    textfont=dict(color=C["text"])))
f.update_layout(**BASE, height=400,
    title=dict(text="<b>Reconstruction Error vs Bottleneck Size</b>", font=dict(size=16)),
    xaxis_title='Bottleneck k', yaxis_title='MSE')
style(f)
figs["ae_bottleneck"] = f

# Training loss curves (linear vs nonlinear)
# Reuse losses computed in the notebook
np.random.seed(42)
We_lin = np.random.randn(2, 5) * 0.1
Wd_lin = np.random.randn(5, 2) * 0.1
We_relu = np.random.randn(2, 5) * 0.1
be_relu = np.zeros(2)
Wd_relu = np.random.randn(5, 2) * 0.1
bd_relu = np.zeros(5)

losses_lin_d, losses_relu_d = [], []
for ep in range(200):
    el, enl = 0, 0
    for ii in range(len(X_ae_data)):
        xi = X_ae_data[ii]
        # Linear
        zl = We_lin @ xi
        xhl = Wd_lin @ zl
        rl = xi - xhl
        el += np.sum(rl**2)
        Wd_lin -= 0.0001 * (-2 * np.outer(rl, zl))
        We_lin -= 0.0001 * (-2 * np.outer(Wd_lin.T @ rl, xi))
        # Nonlinear
        hpre = We_relu @ xi + be_relu
        znl = np.maximum(0, hpre)
        xhnl = Wd_relu @ znl + bd_relu
        rnl = xi - xhnl
        enl += np.sum(rnl**2)
        dxh = -2 * rnl
        Wd_relu -= 0.0001 * np.outer(dxh, znl)
        bd_relu -= 0.0001 * dxh
        dz = Wd_relu.T @ dxh
        dhpre = dz * (hpre > 0).astype(float)
        We_relu -= 0.0001 * np.outer(dhpre, xi)
        be_relu -= 0.0001 * dhpre
    losses_lin_d.append(el / len(X_ae_data))
    losses_relu_d.append(enl / len(X_ae_data))

f = go.Figure()
f.add_trace(go.Scatter(x=list(range(200)), y=losses_lin_d, mode='lines',
    name='Linear AE', line=dict(color=C["primary"], width=2.5)))
f.add_trace(go.Scatter(x=list(range(200)), y=losses_relu_d, mode='lines',
    name='Nonlinear AE (ReLU)', line=dict(color=C["accent"], width=2.5)))
f.update_layout(**BASE, height=400,
    title=dict(text="<b>Training Loss: Linear vs ReLU Autoencoder</b>", font=dict(size=16)),
    xaxis_title='Epoch', yaxis_title='MSE',
    legend=dict(font=dict(color=C["text"])))
style(f)
figs["ae_curves"] = f

# ================================================================
# TAB 3: COMPARISON (MDS vs PCA)
# ================================================================

np.random.seed(42)
X_comp = np.random.randn(60, 6)
X_comp_c = X_comp - X_comp.mean(axis=0)

# PCA
from sklearn.decomposition import PCA as PCA_sk
pca_comp = PCA_sk(n_components=2)
Z_pca_comp = pca_comp.fit_transform(X_comp_c)

# MDS
D_comp = squareform(pdist(X_comp_c))
n_comp = len(D_comp)
J_comp = np.eye(n_comp) - np.ones((n_comp, n_comp)) / n_comp
B_comp = -0.5 * J_comp @ (D_comp**2) @ J_comp
ev_comp, evec_comp = np.linalg.eigh(B_comp)
idx_comp = np.argsort(ev_comp)[::-1]
ev_comp = ev_comp[idx_comp]
evec_comp = evec_comp[:, idx_comp]
Z_mds_comp = evec_comp[:, :2] @ np.diag(np.sqrt(np.maximum(ev_comp[:2], 0)))

# Align signs
for col in range(2):
    if np.corrcoef(Z_pca_comp[:, col], Z_mds_comp[:, col])[0, 1] < 0:
        Z_mds_comp[:, col] *= -1

f = make_subplots(rows=1, cols=2, subplot_titles=['<b>PCA Embedding</b>', '<b>Classical MDS Embedding</b>'],
    horizontal_spacing=0.12)
f.add_trace(go.Scatter(x=Z_pca_comp[:, 0], y=Z_pca_comp[:, 1], mode='markers',
    marker=dict(size=6, color=C["primary"], opacity=0.8), name='PCA'), row=1, col=1)
f.add_trace(go.Scatter(x=Z_mds_comp[:, 0], y=Z_mds_comp[:, 1], mode='markers',
    marker=dict(size=6, color=C["accent"], opacity=0.8), name='MDS'), row=1, col=2)
f.update_layout(**BASE, height=420, showlegend=False,
    title=dict(text="<b>MDS \u2261 PCA</b> (Euclidean distances on centered data)", font=dict(size=16)))
style(f)
figs["comparison"] = f

# ================================================================
# ASSEMBLE DASHBOARD
# ================================================================

def div(key, first=False):
    return figs[key].to_html(full_html=False,
        include_plotlyjs="cdn" if first else False,
        config={"displayModeBar": True, "displaylogo": False})

card = lambda inner: f"<div class='card'>{inner}</div>"

tabs = {
    "MDS": card(div("city_map", first=True)) + card(div("stress")),
    "Autoencoder": card(div("ae_curves")) + card(div("ae_bottleneck")),
    "Comparison": card(div("comparison")),
}

nav = "".join(
    f"<button class='tab-btn{' active' if i==0 else ''}' onclick=\"showTab(event,'tab{i}')\">"
    + name + "</button>" for i, name in enumerate(tabs))
panels = "".join(
    f"<div id='tab{i}' class='tab-panel' style='display:{'block' if i==0 else 'none'}'>{c}</div>"
    for i, c in enumerate(tabs.values()))

html = f"""<div id="dash"><style>
  #dash {{background:{C['bg']};border-radius:14px;padding:22px;font-family:Inter,'Segoe UI',sans-serif;}}
  #dash .dash-header {{display:flex;justify-content:space-between;align-items:baseline;margin-bottom:16px;}}
  #dash h1 {{color:{C['text']};font-size:21px;margin:0;}}
  #dash .sub {{color:{C['muted']};font-size:12px;}}
  #dash .tab-bar {{display:flex;gap:6px;border-bottom:1px solid {C['grid']};margin-bottom:18px;}}
  #dash .tab-btn {{background:none;border:none;color:{C['muted']};padding:10px 18px;font-size:13.5px;font-weight:600;cursor:pointer;border-bottom:2.5px solid transparent;}}
  #dash .tab-btn:hover {{color:{C['text']};}}
  #dash .tab-btn.active {{color:{C['accent']};border-bottom-color:{C['accent']};}}
  #dash .card {{background:{C['card']};border:1px solid {C['grid']};border-radius:12px;padding:6px;margin-bottom:16px;box-shadow:0 2px 10px rgba(0,0,0,.35);}}
</style>
<div class="dash-header"><h1>MDS & Autoencoders Explorer</h1><span class="sub">BDATA 412 \u00b7 Section 7</span></div>
<div class="tab-bar">{nav}</div>{panels}
<script>function showTab(evt,id){{const d=document.getElementById('dash');d.querySelectorAll('.tab-panel').forEach(p=>p.style.display='none');d.querySelectorAll('.tab-btn').forEach(b=>b.classList.remove('active'));document.getElementById(id).style.display='block';evt.currentTarget.classList.add('active');document.getElementById(id).querySelectorAll('.js-plotly-plot').forEach(g=>Plotly.Plots.resize(g));}}</script></div>"""

display(HTML(html))
print("Dashboard ready \u2014 switch tabs to explore MDS, Autoencoder, and Comparison views.")